# Phase 2X - Cross-validation and a corrected leakage measurement

**Run on:** Kaggle or Colab, free T4. Budget ~60-80 min for 20 training runs.

---

### Why this notebook exists

Two weaknesses in the results so far, both addressed here.

**1. Every previous number came from a single split.** One partition, one seed, no
variance. That is adequate for a project report and thin for a paper: a reviewer will
reasonably ask whether the result is an artefact of one lucky partition. Five-fold
cross-validation replaces point estimates with mean and standard deviation.

**2. The Phase 0 leakage estimate measured the wrong thing.** It took one model, whose
training data was fixed, and evaluated it on an image-level test set and on a grouped
test set. The difference between those numbers conflates two effects: genuine leakage,
and the two test sets simply being of different difficulty.

The correct comparison trains **under** each protocol and evaluates within it:

| | training partition | test partition |
|---|---|---|
| Image-level protocol | images assigned independently | images assigned independently |
| Grouped protocol | whole pseudo-patient groups | whole pseudo-patient groups |

Identical architecture, identical hyper-parameters, identical fold count. The only
difference is whether the partition respects pseudo-patient boundaries. Any gap is then
attributable to the protocol. This mirrors the design used by Yagis et al. for
slice-level splitting in brain MRI.

**Two architectures are evaluated**, a from-scratch Vision Transformer and a pretrained
convolutional backbone, so that any effect can be reported as a property of the
**benchmark** rather than of one model.

### Fixed before running
- **No early stopping.** Epoch counts are fixed per architecture, matched to where the
  Phase 2C runs converged, and applied identically across every protocol and fold. Early
  stopping would require a validation split whose construction differs between
  protocols, which would contaminate the comparison it exists to measure.
- **No model selection.** Every configuration is specified in advance and all results
  are reported, so there is no selection effect to correct for.

## 1. Environment and data

In [ ]:
import subprocess, sys
for pkg in ["opencv-python-headless", "tabulate", "kagglehub", "scipy"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", pkg],
                   check=False)

import tensorflow as tf, keras
gpus = tf.config.list_physical_devices("GPU")
print("tensorflow:", tf.__version__, "| keras:", keras.__version__)
print("GPUs      :", gpus)
if not gpus:
    print("\nNO GPU DETECTED - 20 training runs on CPU is not practical.")
    print("  Colab : Runtime > Change runtime type > T4 GPU")
    print("  Kaggle: Settings > Accelerator > GPU (needs a verified phone number)")

In [ ]:
import os, json, time, shutil, random, gc
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                             balanced_accuracy_score, confusion_matrix)
from scipy import stats

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

IN_KAGGLE = os.path.exists("/kaggle/working")
WORK    = "/kaggle/working" if IN_KAGGLE else "/content"
SCRATCH = "/kaggle/temp"    if IN_KAGGLE else "/content"
try:
    os.makedirs(SCRATCH, exist_ok=True)
except OSError:
    SCRATCH = "/tmp"; os.makedirs(SCRATCH, exist_ok=True)
print("environment:", "Kaggle" if IN_KAGGLE else "Colab", "| outputs ->", WORK)

def get_secret(name):
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return (UserSecretsClient().get_secret(name) or "").strip() or None
        from google.colab import userdata
        return (userdata.get(name) or "").strip() or None
    except Exception as e:
        if "NotebookAccess" in type(e).__name__:
            print(f"  {name}: exists but this notebook lacks access")
        return None

def deliver(zip_base, src_dir):
    path = shutil.make_archive(zip_base, "zip", src_dir)
    print("archive:", path, f"({os.path.getsize(path)/1e6:.1f} MB)")
    if not IN_KAGGLE:
        try:
            from google.colab import files; files.download(path)
        except Exception as e:
            print("download it from the file browser:", e)
    else:
        print("Kaggle: it is under /kaggle/working - use the Output panel")

CLASS_NAMES = ["Benign", "Malignant", "Normal"]
FOLDERS = {"Benign": "Bengin cases", "Malignant": "Malignant cases",
           "Normal": "Normal cases"}
N_FOLDS = 5

RESULTS_DIR = f"{WORK}/fyp_phase2x_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS = {"seed": SEED, "n_folds": N_FOLDS}

def save_json():
    with open(f"{RESULTS_DIR}/results.json", "w") as f:
        json.dump(RESULTS, f, indent=2, default=float)
print("results ->", RESULTS_DIR)

In [ ]:
if get_secret("KAGGLE_API_TOKEN"):
    os.environ["KAGGLE_API_TOKEN"] = get_secret("KAGGLE_API_TOKEN")
elif get_secret("KAGGLE_USERNAME") and get_secret("KAGGLE_KEY"):
    os.environ["KAGGLE_USERNAME"] = get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = get_secret("KAGGLE_KEY")
else:
    raise RuntimeError("No Kaggle credential; add KAGGLE_API_TOKEN to Secrets.")

import kagglehub
DL = kagglehub.dataset_download("hamdallak/the-iqothnccd-lung-cancer-dataset")
cands = [d for d, _, _ in os.walk(DL) if os.path.basename(d) == "Malignant cases"]
DATA_ROOT = os.path.dirname(cands[0])

SPLIT_URL = ("https://raw.githubusercontent.com/haseebkhan9081/"
             "LViT_Vision_Transformer/main/split_seed42.csv")
subprocess.run(["wget", "-q", "-O", f"{SCRATCH}/split_seed42.csv", SPLIT_URL], check=True)
df = pd.read_csv(f"{SCRATCH}/split_seed42.csv")
df["path"] = [os.path.join(DATA_ROOT, FOLDERS[l], f)
              for l, f in zip(df["label"], df["file"])]
assert all(os.path.exists(p) for p in df["path"])

y_all = df["y"].to_numpy()
groups = df["group"].to_numpy()
print("images:", len(df), "| pseudo-patient groups:", df["group"].nunique())
print(df["label"].value_counts().to_string())
RESULTS["data"] = {"n_images": int(len(df)), "n_groups": int(df["group"].nunique())}
save_json()

## 2. The two partitioning protocols

`GroupKFold` guarantees that no pseudo-patient group appears in two folds.
`StratifiedKFold` over images ignores group membership entirely, which is the
convention this paper is measuring. Both produce five folds over the same 1097 images,
so the comparison differs in one respect only.

In [ ]:
def folds_grouped():
    gkf = GroupKFold(n_splits=N_FOLDS)
    return list(gkf.split(df, y_all, groups=groups))

def folds_imagelevel():
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    return list(skf.split(df, y_all))

PROTOCOLS = {"grouped": folds_grouped(), "image-level": folds_imagelevel()}

for name, folds in PROTOCOLS.items():
    spans = [len(set(groups[tr]) & set(groups[te])) for tr, te in folds]
    print(f"{name:12s} groups appearing in BOTH train and test, per fold: {spans}")
print()
print("The grouped protocol must show all zeros. The image-level protocol shows how")
print("many pseudo-patients are split across the boundary, which is the leak itself.")
RESULTS["protocol_check"] = {
    n: [int(len(set(groups[tr]) & set(groups[te]))) for tr, te in f]
    for n, f in PROTOCOLS.items()}
assert all(v == 0 for v in RESULTS["protocol_check"]["grouped"]), \
    "a group spans train and test under the grouped protocol"
save_json()

## 3. Architectures

In [ ]:
def build_vit(size=30, n_classes=3, patch=6, proj=64, heads=8, depth=8):
    resize_to = 72

    class Patches(layers.Layer):
        def __init__(self, ps, **kw):
            super().__init__(**kw); self.ps = ps
        def call(self, x):
            b = tf.shape(x)[0]
            p = tf.image.extract_patches(
                images=x, sizes=[1, self.ps, self.ps, 1],
                strides=[1, self.ps, self.ps, 1], rates=[1, 1, 1, 1], padding="VALID")
            return tf.reshape(p, [b, -1, self.ps * self.ps * x.shape[-1]])

    class PatchEncoder(layers.Layer):
        def __init__(self, n, d, **kw):
            super().__init__(**kw)
            self.n = n; self.proj = layers.Dense(d)
            self.pos = layers.Embedding(input_dim=n, output_dim=d)
        def call(self, x):
            return self.proj(x) + self.pos(tf.range(self.n))

    def mlp(x, units, rate):
        for u in units:
            x = layers.Dropout(rate)(layers.Dense(u, activation=tf.nn.gelu)(x))
        return x

    inp = layers.Input((size, size, 3))
    x = keras.Sequential([
        layers.Rescaling(1.0 / 255.0), layers.Resizing(resize_to, resize_to),
        layers.RandomFlip("horizontal"), layers.RandomRotation(0.05),
        layers.RandomZoom(0.15, 0.15)], name="aug")(inp)
    x = Patches(patch)(x)
    enc = PatchEncoder((resize_to // patch) ** 2, proj)(x)
    for _ in range(depth):
        a = layers.LayerNormalization(epsilon=1e-6)(enc)
        a = layers.MultiHeadAttention(num_heads=heads, key_dim=proj, dropout=0.1)(a, a)
        b = layers.Add()([a, enc])
        c = layers.LayerNormalization(epsilon=1e-6)(b)
        c = mlp(c, [proj * 2, proj], 0.1)
        enc = layers.Add()([c, b])
    r = layers.Flatten()(layers.LayerNormalization(epsilon=1e-6)(enc))
    r = mlp(layers.Dropout(0.5)(r), [2048, 1024], 0.5)
    return keras.Model(inp, layers.Dense(n_classes, activation="softmax")(r))

def build_effnet(size=224, n_classes=3):
    inp = layers.Input((size, size, 3))
    x = keras.Sequential([layers.RandomFlip("horizontal"),
                          layers.RandomRotation(0.05),
                          layers.RandomZoom(0.15, 0.15)], name="aug")(inp)
    base = keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet", input_shape=(size, size, 3))
    base.trainable = True
    for layer in base.layers[:-20]:
        layer.trainable = False
    x = base(keras.applications.efficientnet.preprocess_input(x), training=False)
    x = layers.Dropout(0.3)(layers.GlobalAveragePooling2D()(x))
    return keras.Model(inp, layers.Dense(n_classes, activation="softmax")(x))

# Epochs fixed to where the Phase 2C runs converged. No early stopping: it would
# need a validation split whose construction differs between protocols, which would
# contaminate the very comparison this notebook exists to make.
ARCHS = {
    "ViT-30 (from scratch)": {"build": lambda: build_vit(30), "size": 30,
                              "epochs": 60, "batch": 32, "lr": 1e-3},
    "EfficientNetB0 (pretrained)": {"build": lambda: build_effnet(224), "size": 224,
                                    "epochs": 30, "batch": 16, "lr": 1e-4},
}

In [ ]:
CACHE = {}
def images_at(size):
    if size not in CACHE:
        X = np.empty((len(df), size, size, 3), np.float32)
        for i, p in enumerate(tqdm(df["path"], desc=f"load {size}px", leave=False)):
            X[i] = np.asarray(Image.open(p).convert("RGB").resize((size, size),
                                                                  Image.BILINEAR),
                              np.float32)
        CACHE[size] = X
    return CACHE[size]

def evaluate(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], average=None, zero_division=0)
    return {"accuracy": float(accuracy_score(y_true, y_pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
            "macro_f1": float(f1.mean()),
            "f1_benign": float(f1[0]), "f1_malignant": float(f1[1]),
            "f1_normal": float(f1[2])}

## 4. The factorial run

2 protocols x 2 architectures x 5 folds. Class weights are computed from each fold's
own training partition, never from the whole dataset, which would itself leak.

In [ ]:
rows = []
t_start = time.time()

for arch_name, cfg in ARCHS.items():
    X = images_at(cfg["size"])
    for proto_name, folds in PROTOCOLS.items():
        for k, (tr, te) in enumerate(folds):
            keras.backend.clear_session()
            tf.random.set_seed(SEED + k)

            counts = np.bincount(y_all[tr], minlength=3)
            cw = {i: float(len(tr) / (3 * c)) if c else 0.0
                  for i, c in enumerate(counts)}

            model = cfg["build"]()
            model.compile(optimizer=keras.optimizers.Adam(cfg["lr"]),
                          loss="sparse_categorical_crossentropy",
                          metrics=["accuracy"])
            t0 = time.time()
            model.fit(X[tr], y_all[tr], epochs=cfg["epochs"], batch_size=cfg["batch"],
                      class_weight=cw, verbose=0)
            m = evaluate(y_all[te], model.predict(X[te], verbose=0).argmax(1))
            m.update({"architecture": arch_name, "protocol": proto_name, "fold": k,
                      "n_train": int(len(tr)), "n_test": int(len(te)),
                      "minutes": (time.time() - t0) / 60})
            rows.append(m)
            print(f"{arch_name[:22]:22s} | {proto_name:11s} | fold {k} | "
                  f"acc {m['accuracy']:.3f}  bal {m['balanced_accuracy']:.3f}  "
                  f"F1 {m['macro_f1']:.3f}  ({m['minutes']:.1f}m)")
            del model
            gc.collect()
    del CACHE[cfg["size"]]
    gc.collect()

cv = pd.DataFrame(rows)
cv.to_csv(f"{RESULTS_DIR}/cv_per_fold.csv", index=False)
print(f"\ntotal {(time.time()-t_start)/60:.1f} min over {len(cv)} runs")
RESULTS["per_fold"] = cv.to_dict("records")
save_json()

## 5. Cross-validated performance

In [ ]:
summary = (cv.groupby(["architecture", "protocol"])
             .agg(acc_mean=("accuracy", "mean"), acc_std=("accuracy", "std"),
                  bal_mean=("balanced_accuracy", "mean"),
                  bal_std=("balanced_accuracy", "std"),
                  f1_mean=("macro_f1", "mean"), f1_std=("macro_f1", "std"))
             .round(4).reset_index())
print(summary.to_string(index=False))
summary.to_csv(f"{RESULTS_DIR}/cv_summary.csv", index=False)
RESULTS["summary"] = summary.to_dict("records")
save_json()

## 6. The leakage effect, per architecture

For each architecture, the difference between the two protocols' mean scores, with a
95\% confidence interval from Welch's t-test on the two sets of five folds. A positive
difference means the image-level protocol scored **higher**, which is the direction
leakage predicts.

In [ ]:
effects = []
for arch in cv["architecture"].unique():
    a = cv[(cv.architecture == arch) & (cv.protocol == "image-level")]
    b = cv[(cv.architecture == arch) & (cv.protocol == "grouped")]
    for metric in ["accuracy", "balanced_accuracy", "macro_f1"]:
        x, y = a[metric].to_numpy(), b[metric].to_numpy()
        d = x.mean() - y.mean()
        t, p = stats.ttest_ind(x, y, equal_var=False)
        se = np.sqrt(x.var(ddof=1) / len(x) + y.var(ddof=1) / len(y))
        dfree = (x.var(ddof=1)/len(x) + y.var(ddof=1)/len(y))**2 / (
            (x.var(ddof=1)/len(x))**2/(len(x)-1) + (y.var(ddof=1)/len(y))**2/(len(y)-1))
        crit = stats.t.ppf(0.975, dfree)
        effects.append({"architecture": arch, "metric": metric,
                        "image_level": round(float(x.mean()), 4),
                        "grouped": round(float(y.mean()), 4),
                        "difference": round(float(d), 4),
                        "ci95_low": round(float(d - crit * se), 4),
                        "ci95_high": round(float(d + crit * se), 4),
                        "p_value": round(float(p), 5)})

eff = pd.DataFrame(effects)
print(eff.to_string(index=False))
eff.to_csv(f"{RESULTS_DIR}/leakage_effect.csv", index=False)
with open(f"{RESULTS_DIR}/leakage_effect.md", "w") as f:
    f.write(eff.to_markdown(index=False))
RESULTS["leakage_effect"] = eff.to_dict("records")
save_json()

print()
print("Reading: a positive difference whose 95% CI excludes zero indicates the")
print("image-level protocol inflates the score. If the effect holds for BOTH")
print("architectures, it is a property of the benchmark rather than of one model.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, metric in zip(axes, ["accuracy", "balanced_accuracy", "macro_f1"]):
    labels, pos = [], 0
    for arch in cv["architecture"].unique():
        for proto, colour in [("image-level", "tab:red"), ("grouped", "tab:blue")]:
            v = cv[(cv.architecture == arch) & (cv.protocol == proto)][metric]
            ax.errorbar(pos, v.mean(), yerr=v.std(), fmt="o", capsize=4, color=colour)
            ax.scatter([pos] * len(v), v, alpha=.35, s=14, color=colour)
            labels.append(f"{arch.split()[0][:8]}\n{proto}"); pos += 1
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=7)
    ax.set_title(metric); ax.grid(alpha=.3, axis="y"); ax.set_ylim(0, 1)
plt.suptitle(f"{N_FOLDS}-fold cross-validation: image-level (red) vs grouped (blue)",
             fontsize=10)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig_protocol_effect.png", dpi=150)
plt.show()

In [ ]:
tbl = summary.copy()
tbl["accuracy"] = tbl.apply(lambda r: f"{r.acc_mean:.3f} +/- {r.acc_std:.3f}", axis=1)
tbl["balanced acc"] = tbl.apply(lambda r: f"{r.bal_mean:.3f} +/- {r.bal_std:.3f}", axis=1)
tbl["macro-F1"] = tbl.apply(lambda r: f"{r.f1_mean:.3f} +/- {r.f1_std:.3f}", axis=1)
final = tbl[["architecture", "protocol", "accuracy", "balanced acc", "macro-F1"]]
print(final.to_string(index=False))
with open(f"{RESULTS_DIR}/cv_table.md", "w") as f:
    f.write(final.to_markdown(index=False))
RESULTS["final_table"] = final.to_dict("records")
save_json()

print(sorted(os.listdir(RESULTS_DIR)))
deliver(f"{WORK}/fyp_phase2x_bundle", RESULTS_DIR)

## 7. Output and validity checks

`fyp_phase2x_bundle.zip` contains per-fold scores, the cross-validated summary, the
leakage effect with confidence intervals, and the comparison figure.

Conditions that invalidate the run:

- **The grouped protocol must show zero shared groups per fold**, asserted in section 2.
  If it does not, the "clean" arm is not clean and neither number means anything.
- **Class weights must come from each fold's own training partition.** Computing them
  over the whole dataset would leak test-set class distribution into training.
- **No early stopping and no model selection.** Every configuration is fixed in advance
  and every result is reported, so there is no selection effect to correct for.

Interpreting the outcome:

- **A positive difference for both architectures, with confidence intervals excluding
  zero**, supports the claim that image-level partitioning inflates results on this
  benchmark irrespective of model.
- **An effect for only one architecture** is a weaker and more interesting result, and
  is reported as such rather than generalised.
- **No effect** would contradict the Phase 0 estimate. In that case the Phase 0 number
  was measuring test-set difficulty rather than leakage, and this notebook's design is
  the trustworthy one. That outcome gets reported too.